# Availability predictor: offline evaluation (S6.2)

The predictor (S6.1, `coordinator/idlemesh_coord/predictor/`) estimates, for each node and each hour of the
week, the chance that a node which is idle now stays idle for the next N minutes. This notebook fits it on
the days before a split and scores the days after it:

- **Brier score**: the mean of (p - y)^2. 0 is perfect; 0.25 is a constant 0.5.
- **Skill**: 1 - Brier / Brier(baseline), against the fleet's single rate and each node's own rate. Above 0
  means the hour of the week helps.
- **Calibration**: within each band of predictions, how often nodes really stayed idle.

Set `DB` to a coordinator database to use real data. The database is copied first and never changed. With
`DB = None` a synthetic fleet is used (`predictor/synthetic.py`). It is **not** real data: it only shows that
the method works. The Phase 1 exit (S6.6) needs at least two weeks of real samples.

The same numbers come from `python experiments/analysis/predictor_eval.py --synthetic` (or `--db PATH`).
Only the standard library is needed. matplotlib, if installed, draws the calibration curve.

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd()))  # run from experiments/analysis/
from predictor_eval import markdown, run

DB = None            # e.g. Path.home() / ".local/share/idlemesh/coordinator.sqlite3"
TRAIN_DAYS, TEST_DAYS = 28, 7
report = run(DB, TRAIN_DAYS, TEST_DAYS)

In [ ]:
from IPython.display import Markdown, display

display(Markdown(markdown(report)))

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed; the calibration table above has the same numbers.")

if plt:
    fig, ax = plt.subplots(figsize=(5, 5))
    for r in report["results"]:
        cal = [b for b in r["calibration"] if b["n"] >= 100]
        ax.plot([b["mean_p"] for b in cal], [b["observed"] for b in cal], marker="o", label=f"{r['horizon_min']} min")
    ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="calibrated")
    ax.set_xlabel("predicted chance of staying idle")
    ax.set_ylabel("observed rate")
    ax.set_title(f"Calibration ({report['label']})")
    ax.legend()
    plt.show()

## Reading the results

- An **office** routine (busy 09:00 to 17:00 on weekdays) is what the hour of the week captures, so skill
  against the node's own rate is high.
- A **lab** machine, used at random times, and a **server**, idle nearly always, gain little or nothing from
  the hour of the week. Small negative skill there is the price of splitting a node's data into 168 hours.
  The prior (`predictor.prior_strength`) keeps it small.
- With real data, compare the fleet Brier score here with the live figure on the Grafana panel "Brier score
  by node" (S6.4). A live figure much worse than this offline one means the routines have changed.